# Using the CE model in prediction mode

In this notebook, we want to just construct the convex hull for a larger data set compared to the original one. 

TASK:
- Try Do you find any complex structures containing a large number of atoms that are more stable at 0K?

In [ ]:
from icet.tools import enumerate_structures
from icet.core.cluster_expansion import ClusterExpansion
from pathlib import Path
import numpy as np
from tqdm import tqdm

# First, we load the desired model
chemical_symbols = ["Cu", "Au"]

max_atom_num = 8

base_path = Path.cwd().parents[1]
struct_path = (
    base_path
    / "data"
    / f"CE_dataset_{chemical_symbols[0]}{chemical_symbols[1]}"
)

ce = ClusterExpansion.read(struct_path / f"ce_model_{max_atom_num}.ce")

# we enumerate structures up to 12 atoms in size
enumerated_structures = list(
    enumerate_structures(
        structure=ce.primitive_structure,
        sizes=range(1, 13),
        chemical_symbols=ce.chemical_symbols,
        niggli_reduce=False,
    )
)

# now, we evaluate the structures
model_energies = []
species_fractions = []
# we store the number of atoms in each structure
nats = []
for struct in tqdm(enumerated_structures, total=len(enumerated_structures)):
    model_energies.append(ce.predict(struct))
    species_fractions.append(
        np.sum(struct.symbols == chemical_symbols[1]) / len(struct)
    )
    nats.append(len(struct))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import cm
from icet.tools import ConvexHull

model_energies = np.asarray(model_energies)
species_fractions = np.asarray(species_fractions)
unique_nats = np.asarray(nats)

unique_nats = np.unique(nats)
cmap = plt.get_cmap("plasma")
for nid, num_atoms in enumerate(unique_nats):
    indices = np.where(num_atoms == nats)[0]
    plt.plot(
        species_fractions[indices],
        model_energies[indices],
        "o",
        color=cmap(nid / len(unique_nats)),
        label=f"{num_atoms} atoms",
        fillstyle="none",
        zorder=len(unique_nats) - nid,
    )
hull_model = ConvexHull(species_fractions, model_energies)
plt.plot(
    hull_model.concentrations,
    hull_model.energies,
    "-x",
    color="red",
    label="CE convex hull",
)

plt.xlabel(f"{chemical_symbols[1]} fraction")
plt.ylabel("Formation energy / eV/atom")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.3), ncols=4)
plt.show()